In [ ]:
"""
Beach wave collision simulation
================================
Model: 2D scalar wave equation implemented via a pseudodifferential
       operator (psipy), with anisotropy, bottom friction, and Coriolis.

Physical setup:
  Two water blades travel toward each other along the x-axis.
  Their collision generates a lateral wave propagating in the y direction,
  mimicking the swash/backwash interaction observed on gently sloping beaches.

Geometry:
  x ∈ [-Lx/2, Lx/2]  — cross-shore direction
  y ∈ [-Ly/2, Ly/2]  — longshore direction (periodic boundaries)

  Blade 1: starts at x = +X1, travels in -x direction (angle = 0, parallel to y)
  Blade 2: starts at x = -X2, travels in +x direction (angle = ALPHA_R, slightly tilted)

Equation solved:
  ∂²u/∂t² = -psiOp(a(ξ), u) - GAMMA·∂u/∂t

  where the principal symbol encodes wave propagation, anisotropy, and Coriolis:
  a(ξ) = C²·(A_XX·ξ² + 2·A_XY·ξη + A_YY·η²) + F_CORIOLIS·ξη

Physical effects:
  - Anisotropy (A_XX, A_YY, A_XY): different propagation speeds in x vs y
  - Coriolis (F_CORIOLIS): deflects wave energy sideways (Northern hemisphere: rightward)
  - Bottom friction (GAMMA): dissipates energy over time (blades decay realistically)

Parameter summary:
  C_SQUARED          wave speed squared (m²/s²)
  ALPHA_DEG          angle between the two blades (degrees)
  LAMBDA             blade characteristic wavelength, controls blade width (m)
  BLADE_WIDTH_RATIO  sigma = LAMBDA / BLADE_WIDTH_RATIO  (larger → narrower blade)
  HEAVISIDE_SHARPNESS  steepness of the blade leading edge  (larger → sharper front)
  SPEED_FACTOR       scales initial velocity  (1.0 = full WKB speed)
  GAMMA              bottom friction coefficient (s⁻¹)
  F_CORIOLIS         Coriolis parameter f = 2Ω·sin(latitude) (s⁻¹)
  A_XX, A_YY, A_XY   anisotropy matrix coefficients
"""

from solver import *
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML


In [ ]:
# ── 1. Physical and simulation parameters ────────────────────────────────────

# ── Wave speed ──
C_SQUARED = 1.0        # constant c² (m²/s²); replace with G*ALPHA*(x+X0) for variable speed

# ── Wave / blade properties ──
LAMBDA             = 4.0   # characteristic wavelength controlling blade width (m)
BLADE_WIDTH_RATIO  = 6.0   # sigma = LAMBDA / BLADE_WIDTH_RATIO  (larger → narrower)
HEAVISIDE_SHARPNESS = 10.0 # steepness of the blade leading edge  (larger → sharper front)
SPEED_FACTOR       = 0.5   # initial velocity scale  (1.0 = full WKB, <1 = slower)

# ── Blade geometry ──
ALPHA_DEG          = 6.0   # tilt angle of blade 2 relative to blade 1 (degrees)
ALPHA_R            = np.radians(ALPHA_DEG)
BLADE1_ANGLE       = 0.0   # blade 1: parallel to y-axis, travels in -x
BLADE2_ANGLE       = ALPHA_R  # blade 2: tilted by ALPHA_R, travels in +x

# ── Anisotropy matrix  A = [[A_XX, A_XY], [A_XY, A_YY]] ──
# symbol_wave = C² · (A_XX·ξ² + 2·A_XY·ξη + A_YY·η²)
# A_XX > A_YY : faster cross-shore than longshore propagation
# A_XY ≠ 0   : couples x and y directions
A_XX = 1.0   # cross-shore weight
A_YY = 1.0   # longshore weight  (set < 1 to slow down longshore propagation)
A_XY = 0.1   # off-diagonal coupling (0 = principal axes aligned with x, y)

# ── Dissipation and Coriolis ──
GAMMA      = 0.1    # bottom friction (s⁻¹); 0 = no decay, 0.05 = moderate decay
F_CORIOLIS = 0.1   # Coriolis parameter (s⁻¹); 0 = off, >0 = Northern hemisphere
F_VORTICITY = 0.1

# ── Velocity balance between the two blades ──
# Compensates for amplitude differences between straight and angled blades.
VELOCITY_SCALE_BLADE2 = 3


In [ ]:
# ── 2. Grid setup ────────────────────────────────────────────────────────────
Lx, Ly   = 10.0, 14.0
# Nx, Ny = 32, 32 
Nx, Ny = 64, 64    
# Nx, Ny = 128, 128    
# Nx, Ny = 256, 256
Lt, Nt   = 20.0, 800
# Lt, Nt   = 30.0, 1000
n_frames = 400

SIGMA = LAMBDA / BLADE_WIDTH_RATIO
X1 =  Lx / 2.0 - SIGMA
X2 = -Lx / 2.0 + SIGMA

xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
# We MUST match this convention so the solver receives the correct array orientation.
xx, yy = np.meshgrid(xs_1d, ys_1d)   # shape (Nx, Ny)

In [ ]:
# ── 3. SymPy symbols and principal symbol ────────────────────────────────────

x, y, t  = sp.symbols('x y t', real=True)
xi, eta  = sp.symbols('xi eta', real=True)
u_func   = Function('u')
u        = u_func(t, x, y)

# Full principal symbol:
#   a(ξ) = C²·(A_XX·ξ² + 2·A_XY·ξη + A_YY·η²)   ← wave propagation + anisotropy
#          + F_CORIOLIS·ξη                          ← Coriolis deflection
symbol_wave     = C_SQUARED * (A_XX * xi**2 + 2*A_XY * xi*eta + A_YY * eta**2)
symbol_coriolis = F_CORIOLIS * xi * eta
symbol_vorticity = F_VORTICITY * (x* eta - y * xi)
symbol_num      = symbol_wave + symbol_coriolis + symbol_vorticity

print("Principal symbol:")
print("  a(ξ) =", symbol_num)


In [ ]:
# ── 4. Wave equation ─────────────────────────────────────────────────────────
#
#   ∂²u/∂t² = -psiOp(a(ξ), u) - GAMMA·∂u/∂t
#
#   Term 1: -psiOp(a(ξ), u)  — wave propagation, anisotropy, Coriolis (in symbol)
#   Term 2: -GAMMA·∂u/∂t     — bottom friction (energy dissipation, not in symbol
#                               because it is a zeroth-order term in space)

gamma    = sp.Symbol('gamma', positive=True)
equation = sp.Eq(
    diff(u, t, 2),
    -psiOp(symbol_num, u) - gamma * diff(u, t)
)
equation_num = equation.subs({gamma: GAMMA})

print("Equation:")
print(f"  ∂²u/∂t² = -psiOp(a(ξ), u) - {GAMMA}·∂u/∂t")


In [ ]:
# ── 5. Initial conditions ────────────────────────────────────────────────────

def _blade(xx, yy, x_center, sign, angle=0.0):
    """
    Single water blade localised around x_center.
    Grid convention: indexing='xy'  →  axis 0 = y, axis 1 = x.
    """
    nx, ny = np.cos(angle), np.sin(angle)
    # x varies along axis=1, y along axis=0 → formula unchanged
    dist      = nx * (xx - x_center) + ny * yy
    sigma     = LAMBDA / BLADE_WIDTH_RATIO
    gauss     = np.exp(-(dist**2) / (2.0 * sigma**2))
    k_h       = HEAVISIDE_SHARPNESS / sigma
    arg       = np.clip(k_h * sign * dist, -500, 500)
    heaviside = 1.0 / (1.0 + np.exp(arg))
    return gauss * heaviside


def _blade_velocity(xx, yy, x_center, sign, angle=0.0):
    """
    Analytical velocity — no np.gradient, no NaN risk.
    Grid convention: indexing='xy'  →  axis 0 = y, axis 1 = x.
    """
    nx, ny   = np.cos(angle), np.sin(angle)
    dist     = nx * (xx - x_center) + ny * yy
    sigma    = LAMBDA / BLADE_WIDTH_RATIO
    k_h      = HEAVISIDE_SHARPNESS / sigma
    arg      = np.clip(k_h * sign * dist, -500, 500)
    gauss    = np.exp(-(dist**2) / (2.0 * sigma**2))
    sigmoid  = 1.0 / (1.0 + np.exp(arg))
    d_gauss   = -(dist / sigma**2) * gauss
    d_sigmoid = sign * k_h * sigmoid * (1.0 - sigmoid)
    d_blade   = d_gauss * sigmoid + gauss * d_sigmoid
    c_local   = np.sqrt(max(C_SQUARED, 1e-6))
    return -sign * c_local * d_blade * SPEED_FACTOR


def initial_condition_b(xx, yy):
    return (_blade(xx, yy, X1, sign=-1, angle=BLADE1_ANGLE) +
            _blade(xx, yy, X2, sign=+1, angle=BLADE2_ANGLE))


def initial_velocity_b(xx, yy):
    vel1 = _blade_velocity(xx, yy, X1, sign=-1, angle=BLADE1_ANGLE)
    vel2 = _blade_velocity(xx, yy, X2, sign=+1, angle=BLADE2_ANGLE)
    return vel1 - VELOCITY_SCALE_BLADE2 * vel2

In [ ]:
# ── 6. Solver setup ──────────────────────────────────────────────────────────

solver = PDESolver(equation_num)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet',
    initial_condition=initial_condition_b,
    initial_velocity=initial_velocity_b,
    n_frames=n_frames,
    plot=True,
)


In [ ]:
# ── 7. Solve ─────────────────────────────────────────────────────────────────

frames = solver.solve()


In [ ]:
# ── 8. Visualization ─────────────────────────────────────────────────────────

# Raise the animation size limit to allow large frame counts at high resolution
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None,
    mode='imshow',
)

HTML(ani.to_jshtml())


In [ ]:
ani = solver.animate(
    component='real',
    overlay=None,
    mode='surface',
)

HTML(ani.to_jshtml())